In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from transformers import pipeline

# Matplotlib 한글 폰트 설정 (Windows/Mac/Colab 환경에 맞춰 주석 해제)
# 1. Colab 환경인 경우 (설치 후 런타임 재시작 필요)
# !sudo apt-韩国어 폰트 설치 명령어
# plt.rc('font', family='NanumBarunGothic') 

# 2. Windows 환경인 경우
plt.rc('font', family='Malgun Gothic')

# 3. Mac 환경인 경우
# plt.rc('font', family='AppleGothic')

# 마이너스 기호 깨짐 방지
plt.rcParams['axes.unicode_minus'] = False

ModuleNotFoundError: No module named 'matplotlib'

In [ ]:
# 분석할 파일 경로 지정 (원하시는 파일명으로 변경하세요)
file_path = "뉴진스_좋아요포함_댓글50개_영상50개_20260524_145120.xlsx - YouTube Comments.csv"
df = pd.read_csv(file_path)

# 1. 결측치 제거
df = df.dropna(subset=['Comment']).reset_index(drop=True)

# 2. 'Likes' 컬럼 정제: 숫자만 추출하여 int로 변환
def clean_likes(text):
    if pd.isna(text): return 0
    # "다른 사용자 202,760명과 함께..." 형태에서 숫자만 추출
    numbers = re.sub(r'[^0-9]', '', str(text))
    return int(numbers) if numbers else 0

df['Clean_Likes'] = df['Likes'].apply(clean_likes)

# 3. 'Comment' 텍스트 정제: 줄바꿈 제거 및 다중 공백 처리
df['Clean_Comment'] = df['Comment'].apply(lambda x: re.sub(r'\s+', ' ', str(x).strip()))

print(f"총 {len(df)}개의 댓글 데이터가 준비되었습니다.")
display(df[['Video Title', 'Clean_Likes', 'Clean_Comment']].head())

In [ ]:
# 한국어 감정 분석 파이프라인 로드
# device=0 은 GPU 사용 (환경에 GPU가 없다면 -1로 설정하세요)
sentiment_analyzer = pipeline(
    "text-classification", 
    model="monologg/koelectra-small-v2-nsmc", 
    device=-1  
)

# 감정 분석 수행 함수
def analyze_sentiment(text):
    # 모델의 최대 입력 토큰 길이(512)를 고려해 텍스트를 적당히 자름
    truncated_text = text[:500] 
    try:
        result = sentiment_analyzer(truncated_text)[0]
        # LABEL_1은 긍정(Positive), LABEL_0은 부정(Negative)
        label = "Positive" if result['label'] == 'LABEL_1' else "Negative"
        score = result['score']
        return pd.Series([label, score])
    except Exception as e:
        return pd.Series(["Neutral", 0.0])

print("감정 분석을 시작합니다. 데이터 양에 따라 시간이 소요될 수 있습니다...")

# apply를 통해 전체 댓글 감정 분석 적용
df[['Sentiment', 'Confidence']] = df['Clean_Comment'].apply(analyze_sentiment)

display(df[['Clean_Comment', 'Sentiment', 'Confidence']].head())

In [ ]:
# 영상(Video Title)별 긍정/부정 댓글 수 집계
sentiment_counts = df.groupby(['Video Title', 'Sentiment']).size().unstack(fill_value=0)

# 전체 댓글 수 대비 긍정/부정 비율 계산
sentiment_ratio = sentiment_counts.div(sentiment_counts.sum(axis=1), axis=0) * 100

# 시각화: 영상별 긍/부정 비율 누적 막대 그래프
plt.figure(figsize=(12, 8))

# 인덱스(비디오 제목)가 너무 길면 잘리므로 앞 30글자만 표시
video_titles = [title[:30] + '...' if len(title) > 30 else title for title in sentiment_ratio.index]

# Seaborn과 Matplotlib을 혼합하여 누적 바 차트 생성
p1 = plt.barh(video_titles, sentiment_ratio.get('Positive', 0), color='#4C72B0', label='Positive')
p2 = plt.barh(video_titles, sentiment_ratio.get('Negative', 0), left=sentiment_ratio.get('Positive', 0), color='#C44E52', label='Negative')

plt.title('영상별 댓글 감정 비율 (Positive vs Negative)', fontsize=16, pad=15)
plt.xlabel('비율 (%)', fontsize=12)
plt.ylabel('Video Title', fontsize=12)
plt.legend(loc='upper right', bbox_to_anchor=(1.15, 1))
plt.tight_layout()

plt.show()

# 감정 요약 결과 데이터프레임 출력
print("\n[영상별 주된 감정 요약]")
display(sentiment_ratio.round(2))